# Core Demo: config-first conversion authoring

This notebook walks through the intended workflow for arbitrary MCAP data: inspect a bag, draft a reusable spec, edit it declaratively, preflight it, convert it, and inspect the emitted dataset artifacts.


In [ ]:
import json
import sys
from pathlib import Path

demo_dir = Path.cwd()
repo_root = demo_dir.parent if demo_dir.name == "demo" else demo_dir
src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from hephaes import (
    Converter,
    DraftSpecRequest,
    FeatureSpec,
    LabelSpec,
    Profiler,
    RosReader,
    build_draft_conversion_spec,
    dump_conversion_spec,
    inspect_reader,
    load_conversion_spec,
    preflight_conversion_spec,
    stream_tfrecord_rows,
)

bag_files = [
    str(Path("input") / "ros2.mcap"),
]
output_root = Path("./output")


## 1. Profile the bag to get metadata and topics


In [ ]:
topics = []

if not bag_files:
    print("No bag files configured.")
else:
    prof = Profiler(bag_files, max_workers=1)
    metadata_list = prof.profile()
    for meta in metadata_list:
        print(f"File: {meta.file_path}")
        print(f"Path: {meta.path}")
        print(f"ROS version: {meta.ros_version}")
        print(f"Storage format: {meta.storage_format}")
        print(f"File size (bytes): {meta.file_size_bytes}")
        print(f"Start: {meta.start_time_iso} End: {meta.end_time_iso} Duration(s): {meta.duration_seconds}")
        print(f"Message count: {meta.message_count}")
        print("Topics:")
        topics = meta.topics
        for topic in meta.topics:
            print(f"  - {topic.name} ({topic.message_type}): {topic.message_count} messages, {topic.rate_hz} Hz")
        print("-" * 40)

topic_names = [topic.name for topic in topics]
print("Discovered topics:", topic_names)


## 2. Inspect the bag and draft a reusable spec

The point of the converter is not to call a hand-written builder. We inspect sampled payloads, let `hephaes` draft a scaffold, and then replace any low-signal guesses with the actual training fields we want.


In [ ]:
if not bag_files:
    raise RuntimeError("No bag files configured.")

with RosReader.open(bag_files[0]) as reader:
    inspection = inspect_reader(reader, sample_n=32)
    available_topics = list(inspection.topics)
    selected_topics = [
        topic
        for topic in [
            "/nissan/gps/duro/current_pose",
            "/nissan/gps/duro/imu",
            "/nissan/gps/duro/mag",
            "/nissan/vehicle_speed",
            "/nissan/vehicle_steering",
        ]
        if topic in available_topics
    ]
    if not selected_topics:
        raise RuntimeError("The demo bag did not expose any topics to draft from.")

    draft = build_draft_conversion_spec(
        inspection,
        request=DraftSpecRequest(
            selected_topics=selected_topics,
            trigger_topic=selected_topics[0],
            schema_name="core_demo_driving",
            max_features_per_topic=1,
            include_preview=False,
        ),
    )

print("Selected topics:", selected_topics)
print("Draft schema:", draft.spec.schema.model_dump())
print("Draft features:", list(draft.spec.features))
print("Draft row strategy:", draft.spec.row_strategy.model_dump() if draft.spec.row_strategy is not None else None)
print("Draft provenance:", draft.spec.draft_origin.model_dump() if draft.spec.draft_origin is not None else None)


## 3. Edit the draft declaratively

Here we keep the drafted contract, then replace the scaffolded fields with a concrete driving schema for this bag:

- pose position and orientation from `/nissan/gps/duro/current_pose`
- IMU angular velocity and linear acceleration from `/nissan/gps/duro/imu`
- magnetic field from `/nissan/gps/duro/mag`
- vehicle speed from `/nissan/vehicle_speed`
- steering as the label from `/nissan/vehicle_steering`
- `row_timestamp` and `dataset_tag` as runtime context

This is the part you would change for your own dataset without writing a custom converter function.


In [ ]:
demo_features = {
    "pose_x": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/current_pose", "field_path": "pose.position.x"},
        dtype="float64",
        required=True,
    ),
    "pose_y": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/current_pose", "field_path": "pose.position.y"},
        dtype="float64",
        required=True,
    ),
    "pose_z": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/current_pose", "field_path": "pose.position.z"},
        dtype="float64",
        required=True,
    ),
    "pose_qw": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/current_pose", "field_path": "pose.orientation.w"},
        dtype="float64",
        required=True,
    ),
    "pose_qx": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/current_pose", "field_path": "pose.orientation.x"},
        dtype="float64",
        required=True,
    ),
    "pose_qy": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/current_pose", "field_path": "pose.orientation.y"},
        dtype="float64",
        required=True,
    ),
    "pose_qz": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/current_pose", "field_path": "pose.orientation.z"},
        dtype="float64",
        required=True,
    ),
    "imu_ang_vel_x": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/imu", "field_path": "angular_velocity.x"},
        dtype="float64",
        required=False,
    ),
    "imu_ang_vel_y": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/imu", "field_path": "angular_velocity.y"},
        dtype="float64",
        required=False,
    ),
    "imu_ang_vel_z": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/imu", "field_path": "angular_velocity.z"},
        dtype="float64",
        required=False,
    ),
    "imu_lin_acc_x": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/imu", "field_path": "linear_acceleration.x"},
        dtype="float64",
        required=False,
    ),
    "imu_lin_acc_y": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/imu", "field_path": "linear_acceleration.y"},
        dtype="float64",
        required=False,
    ),
    "imu_lin_acc_z": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/imu", "field_path": "linear_acceleration.z"},
        dtype="float64",
        required=False,
    ),
    "mag_x": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/mag", "field_path": "magnetic_field.x"},
        dtype="float64",
        required=False,
    ),
    "mag_y": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/mag", "field_path": "magnetic_field.y"},
        dtype="float64",
        required=False,
    ),
    "mag_z": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/gps/duro/mag", "field_path": "magnetic_field.z"},
        dtype="float64",
        required=False,
    ),
    "vehicle_speed": FeatureSpec(
        source={"kind": "path", "topic": "/nissan/vehicle_speed", "field_path": "data"},
        dtype="float64",
        required=False,
    ),
    "row_timestamp": FeatureSpec(
        source={"kind": "metadata", "key": "timestamp_ns"},
        dtype="int64",
        required=True,
    ),
    "dataset_tag": FeatureSpec(
        source={"kind": "constant", "value": "core-demo"},
        dtype="json",
        required=True,
    ),
}

demo_spec = draft.spec.model_copy(
    update={
        "features": demo_features,
        "validation": draft.spec.validation.model_copy(
            update={
                "preview": True,
                "sample_n": 16,
                "expected_features": list(demo_features),
            }
        ),
        "labels": LabelSpec(
            source={"kind": "path", "topic": "/nissan/vehicle_steering", "field_path": "data"},
        ),
    }
)

spec_json = dump_conversion_spec(demo_spec)
loaded_spec = load_conversion_spec(spec_json)

print("Edited features:", list(loaded_spec.features))
print("Edited row strategy:", loaded_spec.row_strategy.model_dump() if loaded_spec.row_strategy is not None else None)
print("Edited labels:", loaded_spec.labels.model_dump() if loaded_spec.labels is not None else None)
print("Validation config:", loaded_spec.validation.model_dump())
print("Serialized spec preview:")
print(spec_json[:800] + ("..." if len(spec_json) > 800 else ""))


## 4. Preflight the contract before writing shards

Preflight resolves rows, validates feature dtypes and shapes, checks label configuration, and reports missing-data rates before conversion starts.


In [ ]:
with RosReader.open(bag_files[0]) as reader:
    preflight = preflight_conversion_spec(reader, loaded_spec, sample_n=3)

print("Checked records:", preflight.checked_records)
print("Bad records:", preflight.bad_records)
print("Missing feature rates:", preflight.missing_feature_rates)
print("Missing topic rates:", preflight.missing_topic_rates)
print("Label summary:", preflight.label_summary)
print("Preview rows:")
for row in preflight.rows:
    print(row.model_dump())


## 5. Convert and inspect the artifacts

Once the contract looks right, conversion uses the same row construction and validation path we just preflighted.


In [ ]:
spec_output_dir = output_root / "config_first_demo"
spec_files = Converter(
    bag_files,
    None,
    spec_output_dir,
    spec=loaded_spec,
    max_workers=1,
).convert()

output_tfrecord = spec_files[0]
manifest_path = output_tfrecord.with_name(f"{output_tfrecord.stem}.manifest.json")
report_path = output_tfrecord.with_name(f"{output_tfrecord.stem}.report.md")

print("Dataset:", output_tfrecord)
print("Manifest:", manifest_path)
print("Report:", report_path)

manifest = json.loads(manifest_path.read_text())
print("Manifest row strategy:", manifest["conversion"].get("row_strategy"))
print("Manifest draft origin:", manifest["conversion"].get("draft_origin"))
print("Manifest preflight:", manifest["conversion"].get("preflight"))

for i, row in enumerate(stream_tfrecord_rows(output_tfrecord)):
    print(row)
    if i + 1 >= 5:
        break
